# Deploy NVIDIA-Nemotron-3-Super-120B-A12B-NVFP4 on a SageMaker Endpoint


In this notebook, we will show how to deploy the `nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-NVFP4` model on a SageMaker AI endpoint. 

The [NVIDIA-Nemotron-3-Super-120B-A12B-NVFP4](https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-NVFP4) model is a 120-billion-parameter open model with 12 billion active parameters designed to run complex agentic AI systems at scale. This model combines advanced reasoning capabilities to efficiently complete tasks with high accuracy for autonomous agents. Please see the NVIDIA [blog](https://blogs.nvidia.com/blog/nemotron-3-super-agentic-ai/) for more details.

This notebook shows how to deploy this model using the SageMaker Large Model Inference (LMI) [container](https://aws.github.io/deep-learning-containers/reference/available_images/#djl-inference). The LMI container is a first-party model serving container that combines a DJL front end and the popular vLLM backend.

This is a 4-bit quantized model that requires 74 GB for the model weights. Recently, SageMaker AI has added the `g7e` family of [instances](https://aws.amazon.com/ec2/instance-types/g7e/) to the service. In this example, we are going to use a `g7e.2xlarge` instance accelerated by one NVIDIA RTX PRO™ 6000 Blackwell Server Edition GPU, which allows us to deploy the NVFP4 version of the model in a cost-effective way.


In [ ]:
%pip install --upgrade --quiet --no-warn-conflicts boto3

In [ ]:
import time
import re
import json
import boto3
from IPython.display import display, Markdown, clear_output

boto_session = boto3.Session()
region = boto_session.region_name

sm = boto3.client("sagemaker")  # client to intreract with SageMaker
sm_runtime = boto3.client("sagemaker-runtime")  # client to intreract with SageMaker Endpoints

In [ ]:
#
# Helper functions to remove dependency on SageMaker Python SDK
#
def get_sagemaker_role():
    sts = boto3.client('sts')
    response = sts.get_caller_identity()
    assumed_role = response['Arn']
    role = re.sub(r"^(.+)sts::(\d+):assumed-role/(.+?)/.*$", r"\1iam::\2:role/\3", assumed_role)
    return role


def wait_for_endpoint(endpoint_name: str, sleep_time: int=60):
    ind = "."
    progress = f"Waiting for '{endpoint_name}': "
    print(progress)
    
    status = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]
    
    while status == "Creating":
        time.sleep(sleep_time)
        
        status = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]
        
        clear_output(wait=True)
        progress += ind
        print(progress)
  
    print(f"Endpoint: '{endpoint_name}', Status: '{status}'")


def wait_for_ic(ic_name: str, sleep_time: int=60):
    ind = "."
    progress = f"Waiting for '{ic_name}': "
    print(progress)
    
    status = sm.describe_inference_component(InferenceComponentName=ic_name)["InferenceComponentStatus"]
    
    while status == "Creating":
        time.sleep(sleep_time)
        
        status = sm.describe_inference_component(InferenceComponentName=ic_name)["InferenceComponentStatus"]
        
        clear_output(wait=True)
        progress += ind
        print(progress)
  
    print(f"IC: '{ic_name}', Status: '{status}'")

In [ ]:
#
# Overwrite with your role ARN if you are running this notebook outside of SageMaker Studio
#
role = None

if role == None:
    role = get_sagemaker_role()
print(role)

## Common config

In [ ]:
instance = {"type": "ml.g7e.2xlarge", "num_gpu": 1}

model_id = "nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-NVFP4" #model_id = "/opt/ml/model"
model_name = f"model-{time.strftime('%y%m%d-%H%M%S')}"
endpoint_name = model_name
endpoint_config_name = model_name
timeout = 1200
variant_name = "v1"

### LMI configuration

In [ ]:
CONTAINER_VERSION = "0.36.0-lmi22.0.0-cu129"
inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/djl-inference:{CONTAINER_VERSION}"

common_env = {
    "HF_MODEL_ID": model_id,
}
lmi_env = {
    "SERVING_FAIL_FAST": "true",
    "OPTION_ASYNC_MODE": "true",
    "OPTION_TENSOR_PARALLEL_DEGREE": json.dumps(instance["num_gpu"]),
    "OPTION_MAX_MODEL_LEN": "16384",
    "OPTION_KV_CACHE_DTYPE": "fp8", 
    "OPTION_TRUST_REMOTE_CODE": "true",
    "OPTION_ENABLE_AUTO_TOOL_CHOICE": "true",
    "OPTION_TOOL_CALL_PARSER": "qwen3_coder",
    "OPTION_REASONING_PARSER": "deepseek_r1"
}
env = common_env | lmi_env

## Deployment

In [ ]:
_ = sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={
        "Image": inference_image,
        "Environment": env,
    },
)

In [ ]:
_ = sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[
        {
            "VariantName": variant_name,
            "ModelName": model_name,
            "InstanceType": instance["type"],
            "InitialInstanceCount": 1,
            "ContainerStartupHealthCheckTimeoutInSeconds": timeout,
            "InferenceAmiVersion": "al2023-ami-sagemaker-inference-gpu-4-1",
        },
    ],
)

_ = sm.create_endpoint(EndpointName=endpoint_name, 
                       EndpointConfigName=endpoint_config_name)

_ = wait_for_endpoint(endpoint_name)

In [109]:
payload={
    "messages": [
        #{"role": "user", "content": "Name popular places to visit in London?"}
        {"role": "user", "content": "Solve this problem step by step: What is 15% of 240?"}
    ],
}

start_time = time.time()
res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                 Body=json.dumps(payload),
                                 ContentType="application/json")
response = json.loads(res["Body"].read().decode("utf8"))
end_time = time.time()

print(f"✅ Response time: {end_time-start_time:.2f}s\n")
display(Markdown(response["choices"][0]["message"]["content"]))

usage = response["usage"] 
print(f'-----------------------\n{usage}')

✅ Response time: 14.07s




To find 15% of 240, follow these steps:

### Step 1: Convert the percentage to a decimal
- 15% means 15 per 100, so divide by 100:  
  \( 15\% = \frac{15}{100} = 0.15 \).

### Step 2: Multiply the decimal by the number
- Multiply 0.15 by 240:  
  \( 0.15 \times 240 \).

### Step 3: Perform the multiplication
- You can break it down for clarity:  
  - \( 10\% \text{ of } 240 = 240 \times 0.10 = 24 \)  
  - \( 5\% \text{ of } 240 = 240 \times 0.05 = 12 \) (since 5% is half of 10%)  
  - Add them together: \( 24 + 12 = 36 \).  
- Alternatively, multiply directly:  
  \( 0.15 \times 240 = \frac{15}{100} \times 240 = \frac{15 \times 240}{100} = \frac{3600}{100} = 36 \).

### Step 4: State the result
- 15% of 240 is **36**.

**Verification**:  
- 10% of 240 = 24, 20% of 240 = 48, so 15% (midway between 10% and 20%) should be \( \frac{24 + 48}{2} = 36 \), which confirms the answer.

**Final Answer**: 36

-----------------------
{'prompt_tokens': 36, 'total_tokens': 1023, 'completion_tokens': 987, 'prompt_tokens_details': None}


To give you a rough idea of what performance you should expect from this cost-effective configuration, here are the results of running 100 requests with an average input size of 500 tokens and an average output size of 500 tokens:

| Metric | avg | min | max | p1 | p5 | p10 | p25 | p50 | p75 | p90 | p95 | p99 | std |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Input Sequence Length (tokens) | 500.72 | 360 | 620 | 394.65 | 416.7 | 435 | 472.5 | 501 | 526.5 | 576.1 | 586 | 612.08 | 50.26 |
| Inter Token Latency (ms) | 14.14 | 13.76 | 18.55 | 13.93 | 13.95 | 13.95 | 13.97 | 14.09 | 14.11 | 14.29 | 14.32 | 16.23 | 0.51 |
| Output Sequence Length (tokens) | 500.89 | 345 | 688 | 376.68 | 414.4 | 431.9 | 466.75 | 496 | 534.25 | 567 | 589.65 | 675.13 | 56.31 |
| Request Latency (ms) | 7199.98 | 4922.6 | 9818.03 | 5451.44 | 5879.66 | 6310.09 | 6680.63 | 7163.22 | 7719.46 | 8106.53 | 8392.49 | 9548 | 802.7 |
| Time to First Token (ms) | 135.88 | 112.72 | 1202.07 | 113.21 | 113.76 | 114.31 | 116.87 | 121.83 | 124.83 | 126.78 | 130.07 | 483.02 | 112.99 |

## Cleanup

In [110]:
_ = sm.delete_endpoint(EndpointName=endpoint_name)
_ = sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
_ = sm.delete_model(ModelName=model_name)